## Regras de Classificação das Estações

Este notebook aplica regras de enriquecimento e classificação sobre o dataset de estações, conforme descrito abaixo:

---

### 1. Classificação de Severidade (OMR → Tipologia)

O campo `severidade_omr` é transformado em uma tipologia operacional com base no seguinte mapeamento:

| Severidade OMR       | Tipologia                                                                 |
|---------------------|--------------------------------------------------------------------------|
| 02 - LEVE           | Ponta ou Repetidora de RF                                                |
| 04 - MODERADO       | Estratégica e Monosite                                                   |
| 08 - SEVERO         | Concentradora com repetidores MW < 20 sites                              |
| 16 - CRITICO        | Com roteadores IPRAN ou com repetidores MW >= 20 sites                   |
| 32 - MUITO CRITICO  | Central, BSC e RNC remotas e GPON                                        |
| (vazio)             | Ponta ou Repetidora de RF                                                |

---

### 2. Classificação de Unidade

A coluna `unidade` é derivada com base no campo `alias_ebt`:

- Se `alias_ebt` estiver preenchido → **Empresarial/Móvel**
- Caso contrário → **Móvel**

---

### 3. Classificação de COW

A coluna `cow` indica se a estação é do tipo móvel (Cell on Wheels):

- Se o nome da estação (`estacao`) contém **"COW"** ou **"CWX"** → **SIM**
- Caso contrário → **NÃO**

---

### 4. Identificação de Licença RF

Uma estação é considerada com licença RF (`licenca_rf = True`) quando:

- `anatel_rf` está preenchido **OU**
- `anatel_nextel_rf` está preenchido

---

### 5. Contagem de Estações com Licença RF por Município

Para cada município, é calculada a quantidade de estações distintas que possuem licença RF:

- Considera apenas estações com `licenca_rf = True`
- A contagem é feita utilizando `nunique(estacao)`

---

### 6. Classificação de Monosite

A coluna `monosite` segue a regra de negócio:

- Se o município possui **EXATAMENTE 1 estação com licença RF** → todas as estações do município são classificadas como **SIM**
- Caso contrário (0 ou mais de 1 estação com licença RF) → todas as estações do município são classificadas como **NÃO**

#### Resumo da regra:

| Qtde de estações com RF no município | Resultado |
|------------------------------------|----------|
| 0                                  | NÃO      |
| 1                                  | SIM      |
| ≥ 2                                | NÃO      |

---

In [27]:
import pandas as pd


pd.set_option('display.max_colwidth', None)

In [28]:
df_estacoes = pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\raw\estacoes.pkl')
df_estacoes.head()

,estacao,cluster,uf,municipio,ibge,alias_ebt,nome_ref_estacao,severidade_omr,finalidade_movel,finalidade_empresarial,finalidade_residencial,anatel_rf,anatel_nextel_rf,latitude,longitude
0,BAFSA37,BA/SE,BA,FEIRA DE SANTANA,2910800,FSA VX,,04 - MODERADO,SITE RF,PPC,,696220962,1011554876,-12.253807,-38.964660
1,BAFSA37_001,BA/SE,BA,FEIRA DE SANTANA,2910800,,SLS - FEIRA DE SANTANA,,SITE RF,,,1016660135,,-12.253044,-38.966417
2,BAFSA37_002,BA/SE,BA,FEIRA DE SANTANA,2910800,,SLS - FEIRA DE SANTANA,,SETOR REMOTO,,,1016268936,,-12.255990,-38.965210
3,BAFSA37_003,BA/SE,BA,FEIRA DE SANTANA,2910800,,,,SITE RF,,,1016688854,,-12.252900,-38.966290
4,BAFSA38,BA/SE,BA,FEIRA DE SANTANA,2910800,FSA SC5,,16 - CRITICO,SITE RF,TER,,699578949,,-12.256380,-38.969304


In [29]:
mapa_tipologias = {
    '02 - LEVE': 'Ponta ou Repetidora de RF',
    '04 - MODERADO': 'Estratégica e Monosite',
    '08 - SEVERO': 'Concentradora com repetidores MW <20 sites',
    '16 - CRITICO': 'Com roteadores IPRAN ou com repetidores MW >= 20 sites',
    '32 - MUITO CRITICO': 'Central, BSC e RNC remotas e GPON',
    '': 'Ponta ou Repetidora de RF'
}

df_estacoes['severidade_omr'] = df_estacoes['severidade_omr'].map(mapa_tipologias)

df_estacoes['alias_ebt_tratado'] = (
    df_estacoes['alias_ebt']
    .fillna('')
    .astype(str)
    .str.strip()
)

df_estacoes['unidade'] = df_estacoes['alias_ebt_tratado'].apply(lambda valor: 'Empresarial/Móvel' if valor != '' else 'Móvel')


df_estacoes['cow'] = (
    df_estacoes['estacao']
    .fillna('')
    .astype(str)
    .str.upper()
    .str.contains('COW|CWX', regex=True)
).map({
    True: 'SIM',
    False: 'NÃO'
})

#classificando monosites
df_estacoes['anatel_rf'] = df_estacoes['anatel_rf'].fillna('').astype(str).str.strip()
df_estacoes['anatel_nextel_rf'] = df_estacoes['anatel_nextel_rf'].fillna('').astype(str).str.strip()
df_estacoes['licenca_rf'] = ((df_estacoes['anatel_rf'] != '') | (df_estacoes['anatel_nextel_rf'] != ''))

#conta quantas estações com licença de RF existem por município
qtd_estacoes_licenca_rf_por_municipio = df_estacoes[df_estacoes['licenca_rf']].groupby('municipio')['estacao'].nunique()
df_estacoes['qtd_estacoes_licenca_rf_municipio'] = df_estacoes['municipio'].map(qtd_estacoes_licenca_rf_por_municipio).fillna(0).astype(int)

#se a estação se encontra em um município que contém uma única estação com licença de RF, todas as estações são monosites
df_estacoes['monosite'] = (
    df_estacoes['qtd_estacoes_licenca_rf_municipio'].eq(1)
).map({
    True: 'SIM',
    False: 'NÃO'
})

df_estacoes = df_estacoes[
    [
        'estacao',
        'severidade_omr',
        'cow',
        'monosite'
    ]
]

df_estacoes.head()

,estacao,severidade_omr,cow,monosite
0,BAFSA37,Estratégica e Monosite,NÃO,NÃO
1,BAFSA37_001,Ponta ou Repetidora de RF,NÃO,NÃO
2,BAFSA37_002,Ponta ou Repetidora de RF,NÃO,NÃO
3,BAFSA37_003,Ponta ou Repetidora de RF,NÃO,NÃO
4,BAFSA38,Com roteadores IPRAN ou com repetidores MW >= 20 sites,NÃO,NÃO


In [30]:
df_indisponibilidade = pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\raw\indisponibilidade.pkl')
df_indisponibilidade = df_indisponibilidade[['estacao', 'mttr']]
df_indisponibilidade = df_indisponibilidade[df_indisponibilidade['estacao'] != '']
df_indisponibilidade['mttr_maior_que_2h'] = (df_indisponibilidade['mttr'] > 7200).map({True: 'SIM', False: 'NÃO'})
df_indisponibilidade = df_indisponibilidade[['estacao', 'mttr_maior_que_2h']]
df_indisponibilidade = df_indisponibilidade.drop_duplicates()
df_indisponibilidade.head()

,estacao,mttr_maior_que_2h
0,PRMOS01,NÃO
1,PRATT12,NÃO
2,PRCCJ40,NÃO
3,PRCSF22,NÃO
4,PRCSFB7,NÃO


In [31]:
df_geradores = pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\raw\geradores.pkl')
df_geradores.head()

,estacao,gerador
0,RJOFRG,REDUNDANTE
1,RJOBOT,REDUNDANTE
2,RJOCPG,REDUNDANTE
3,RJOPRN,SEM GERADOR
4,RJOPIA,REDUNDANTE


In [34]:
df = df_estacoes.merge(df_indisponibilidade, on='estacao', how='left')
df = df.merge(df_geradores, on='estacao', how='left')
#nulos
df['mttr_maior_que_2h'] = df['mttr_maior_que_2h'].fillna('SEM INDISPONIBILIDADE')
df['gerador'] = df['gerador'].fillna('SEM GERADOR')
df['autonomia_projetada'] = ''
df.head(30)

,estacao,severidade_omr,cow,monosite,mttr_maior_que_2h,gerador,autonomia_projetada
0,BAFSA37,Estratégica e Monosite,NÃO,NÃO,NÃO,SEM GERADOR,
1,BAFSA37,Estratégica e Monosite,NÃO,NÃO,SIM,SEM GERADOR,
2,BAFSA37_001,Ponta ou Repetidora de RF,NÃO,NÃO,SEM INDISPONIBILIDADE,SEM GERADOR,
3,BAFSA37_002,Ponta ou Repetidora de RF,NÃO,NÃO,SEM INDISPONIBILIDADE,SEM GERADOR,
4,BAFSA37_003,Ponta ou Repetidora de RF,NÃO,NÃO,SEM INDISPONIBILIDADE,SEM GERADOR,
5,BAFSA38,Com roteadores IPRAN ou com repetidores MW >= 20 sites,NÃO,NÃO,NÃO,REDUNDANTE,
6,BAFSA38,Com roteadores IPRAN ou com repetidores MW >= 20 sites,NÃO,NÃO,SIM,REDUNDANTE,
7,BAFSA38_001,Ponta ou Repetidora de RF,NÃO,NÃO,SEM INDISPONIBILIDADE,SEM GERADOR,
8,BAFSA41,Estratégica e Monosite,NÃO,NÃO,NÃO,SEM GERADOR,
9,BAFSA41,Estratégica e Monosite,NÃO,NÃO,SIM,SEM GERADOR,


In [33]:
df.severidade_omr.unique()

<ArrowStringArray>
[                                'Estratégica e Monosite',
                              'Ponta ou Repetidora de RF',
 'Com roteadores IPRAN ou com repetidores MW >= 20 sites',
             'Concentradora com repetidores MW <20 sites',
                      'Central, BSC e RNC remotas e GPON']
Length: 5, dtype: str